# 02-plan: where the next stores go, CP-SAT against a greedy baseline

Given the demand per **hex** (`00-demand`) and which hexes each **candidate site** reaches (`01-reach`), pick N sites
for new stores so that the **operator** serves as many orders a day as possible, with a placeholder **synthetic network**
of the operator's and two competitor **brands'** existing stores. The model follows
[ADR 0001](../docs/adr/0001-demand-split-by-brand.md): a hex reached by several brands is split equally between the brands,
and the operator assigns each hex to one of its stores within **capacity**. Two plans are compared, a greedy baseline and
CP-SAT, both scored by the same evaluator. The notebook also times the solve and extrapolates it to the full area, since
the product has to return a **plan** in 10 s or less.

The question this notebook answers: **does CP-SAT beat the greedy baseline, and can it solve the full-area default
scenario (N = 10) within 10 s?** Everything runs on the slice at H3 res 9 as the main case, and res 8 where it is cheap.

In [ ]:
import math
import os
import time
from collections import defaultdict
from dataclasses import dataclass, replace

import h3
import matplotlib.pyplot as plt
import numpy as np
import ortools
import pandas as pd
import geopandas as gpd
from matplotlib.lines import Line2D
from matplotlib.axes import Axes
from matplotlib.ticker import NullFormatter
from ortools.sat.python import cp_model
from shapely.geometry import Polygon

from catchment.fetch import DATA

EXPLORE = DATA / "explore"
FIGS = DATA.parent / "notebooks" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

## Assumptions

Every number the plan rests on, in one place. **Verified** means checked against the source on the date shown;
**Unverified** means taken from a secondary report and still to be checked; **Assumption** means a choice with no
external source. The network counts are placeholders (see below), and the capacity's 1.5 factor has no source.

In [ ]:
# --- Assumptions ---------------------------------------------------------
PICK_AND_PACK = 3                       # minutes, fixed
PROMISE, MULTIPLIER = 10, 1.4           # default scenario
N_SLICE, N_AREA = 3, 10                 # new stores: the slice, the full area (default scenario)

# Capacity: 1.5 x the calibrated average orders per store per day (inputs as in 00-demand)
BLINKIT_ORDERS_Q2, Q2_DAYS, BLINKIT_STORES = 398.9e6, 92, 2751
ORDERS_PER_STORE_DAY = BLINKIT_ORDERS_Q2 / Q2_DAYS / BLINKIT_STORES
CAPACITY_FACTOR = 1.5
CAPACITY = CAPACITY_FACTOR * ORDERS_PER_STORE_DAY       # orders per store per day

# Placeholder synthetic network: the slice holds about 10.5 stores' worth of orders (00-demand)
BRAND_COUNTS = {"Operator": 3, "Competitor A": 4, "Competitor B": 3}
SEED = 2026                             # network placement

# Solver and integer scale
SCALE = 100                             # shares and capacity are integers in hundredths of an order
CAP_INT = round(CAPACITY * SCALE)
TIME_LIMIT, WORKERS, SOLVER_SEED = 60, 8, 1
X_GRID, N_GRID = [2.5, 5, 10, 15, 20, 30], [3, 5, 10]   # solve-time grid: X as in 01-reach
FULL_AREA = {9: {"candidates": 1458, "pairs": 54246}, 8: {"candidates": 210, "pairs": 1105}}   # from 01-reach, default scenario

def reachable(ff_min: float | np.ndarray | pd.Series) -> bool | np.ndarray | pd.Series:
    """Hex reachable from a site: free-flow minutes x multiplier + pick-and-pack <= promise."""
    return ff_min * MULTIPLIER + PICK_AND_PACK <= PROMISE + 1e-9

assumptions = pd.DataFrame(
    [
        ("Default scenario", f"Promise {PROMISE} min, congestion multiplier {MULTIPLIER}, pick-and-pack {PICK_AND_PACK} min: a hex is reachable if free-flow min × {MULTIPLIER} + {PICK_AND_PACK} ≤ {PROMISE}, that is free-flow ≤ {(PROMISE - PICK_AND_PACK) / MULTIPLIER:.1f} min", "Decision log, 2026-10-29 (Explore)", "2026-10-29", "Assumption"),
        ("New stores, N", f"{N_SLICE} on the slice; {N_AREA} on the full area (the default scenario for the 10 s bar)", "Decision log, 2026-10-29 (Explore)", "2026-10-29", "Assumption"),
        ("Orders per store per day", f"{BLINKIT_ORDERS_Q2/1e6:.1f} M orders ÷ {Q2_DAYS} days ÷ {BLINKIT_STORES:,} stores = {ORDERS_PER_STORE_DAY:,.0f}", "00-demand: Eternal Ltd shareholders' letter, 21 Oct 2026, p. 13 (orders and stores)", "2026-10-29", "Verified"),
        ("Capacity per store", f"{CAPACITY_FACTOR} × {ORDERS_PER_STORE_DAY:,.0f} = {CAPACITY:,.0f} orders per day, the same for every store of the operator. Adopted here as 00-demand and the decision log said; the 1.5 has no source (headroom over the average)", "Decision log, 2026-10-29 (Explore); issue #5. Rests on the Verified orders per store per day", "2026-10-29", "Assumption"),
        ("Brands and store counts", f"{', '.join(f'{b} {n}' for b, n in BRAND_COUNTS.items())}: {sum(BRAND_COUNTS.values())} stores in the slice. PLACEHOLDER counts, not reported figures", "Slice demand ÷ orders per store per day = 10.5 stores' worth (00-demand), rounded to 10 and split with no market-share source. The generator with sourced counts is stage 2", "2026-10-29", "Assumption"),
        ("Store placement", f"Sampled without replacement from the slice's candidate sites, probability ∝ the demand the site reaches at the default scenario; seed {SEED}; at most one store per site, whatever the brand", "No external source", "-", "Assumption"),
        ("Competitor reach", "A competitor store at a site reaches the same hexes as an operator store there (same promise, multiplier and travel times)", "No external source", "-", "Assumption"),
        ("Demand split", "A hex reached by the operator and k competitor brands gives the operator demand ÷ (1 + k); k is fixed in a scenario", "ADR 0001", "2026-10-29", "Assumption"),
        ("Sites for new stores", "Any slice candidate site with no store on it, of any brand", "No external source", "-", "Assumption"),
        ("Assignment tie-break", "Among assignments that serve the same orders, the one with the shortest total travel time (decides new against cannibalised demand)", "No external source", "-", "Assumption"),
        ("Integer scale", f"Shares and capacity in hundredths of an order (round(orders × {SCALE})), because CP-SAT needs integers. Every metric is computed from these rounded shares", "Modelling choice", "-", "Assumption"),
        ("Solver", f"OR-tools {ortools.__version__} CP-SAT, time limit {TIME_LIMIT} s, {WORKERS} workers, random_seed {SOLVER_SEED}", "Modelling choice", "-", "Assumption"),
    ],
    columns=["Input", "Value", "Source", "Date", "Status"],
)
pd.set_option("display.max_colwidth", None)
display(assumptions.style.hide(axis="index").set_properties(**{"text-align": "left"}))

## The slice at the default scenario

For each resolution: the slice's hexes (`hexes-r{9,8}.parquet`, with demand), the slice's candidate sites
(`candidates-r{9,8}.parquet`, X = 30%, ranked by places count, rank 0 busiest) and the pairs of the travel-time table
(`table-r{9,8}.parquet`) that are reachable at the default scenario, free-flow ≤ 5.0 min. A `Case` holds all of this as
arrays. Sites and hexes are numbered by position; sites are in rank order, so "first" means "busiest".

In [ ]:
@dataclass
class Case:
    res: int
    hexes: pd.Index
    demand: np.ndarray        # orders per day per hex
    sites: pd.Index           # slice candidate sites, in rank order
    rank: np.ndarray          # places-count rank in the whole area, 0 = busiest
    ps: np.ndarray            # reachable pairs: site position
    ph: np.ndarray            # reachable pairs: hex position
    pm: np.ndarray            # reachable pairs: free-flow minutes
    n_area_hexes: int         # hexes in the whole area, to turn X% into a count
    stores: dict[str, list[int]]   # brand -> site positions
    k: np.ndarray             # competitor brands reaching each hex
    share_int: np.ndarray     # operator share of each hex if it serves it, hundredths of an order
    new_sites: np.ndarray     # site positions open for a new store
    cap_int: int              # capacity, hundredths of an order

def load_case(res: int) -> Case:
    hx = pd.read_parquet(EXPLORE / f"hexes-r{res}.parquet")
    n_area = len(hx)
    hx = hx[hx.in_slice]
    cand = pd.read_parquet(EXPLORE / f"candidates-r{res}.parquet")
    cand = cand[cand.in_slice].sort_values("rank")
    tab = pd.read_parquet(EXPLORE / f"table-r{res}.parquet")
    tab = tab[reachable(tab.minutes)]
    spos = pd.Series(np.arange(len(cand)), index=cand.index)
    hpos = pd.Series(np.arange(len(hx)), index=hx.index)
    d = hx["demand"].to_numpy()
    return Case(res, hx.index, d, cand.index, cand["rank"].to_numpy(), spos[tab.candidate].to_numpy(),
                hpos[tab.hex].to_numpy(), tab.minutes.to_numpy(), n_area, {}, np.zeros(len(d), int),
                np.round(d * SCALE).astype(np.int64), np.arange(len(cand)), CAP_INT)

bare = {res: load_case(res) for res in (9, 8)}
rows = {}
for res, C in bare.items():
    per_hex = np.bincount(C.ph, minlength=len(C.hexes))
    rows[f"res {res}"] = {
        "hexes, slice": len(C.hexes),
        "demand, slice (orders/day)": f"{C.demand.sum():,.0f}",
        "candidate sites, slice": len(C.sites),
        "default-scenario pairs (≤ 5.0 min)": f"{len(C.ps):,}",
        "mean hexes reached per site": f"{len(C.ps) / len(C.sites):.1f}",
        "hexes no candidate reaches": int((per_hex == 0).sum()),
        "demand on those hexes (orders/day)": f"{C.demand[per_hex == 0].sum():,.0f}",
        "largest catchment's demand (orders/day)": f"{np.bincount(C.ps, weights=C.demand[C.ph]).max():,.0f}",
    }
display(pd.DataFrame(rows))
print(f"Capacity {CAPACITY:,.0f} orders/day = {CAP_INT:,} hundredths")

## The placeholder synthetic network

The operator and two competitor brands each have existing stores on slice candidate sites. **The counts are
placeholders.** The slice's demand is about 10.5 stores' worth at `ORDERS_PER_STORE_DAY` (`00-demand`), so the slice
gets 10 stores in all: operator 3, Competitor A 4, Competitor B 3. There is no sourced market share behind the split;
it only makes the operator a similar size to the others. Stores are drawn without replacement from the slice's
candidate sites, with probability proportional to the demand a site reaches at the default scenario (brands choose busy
places), from a seeded `numpy` generator; which brand gets which drawn site is a seeded shuffle. No site holds two stores.
Competitors reach the same hexes the operator would from the same site. The real generator, with counts sized from
reported figures and their dates, is stage 2.

For each hex, **k** is the number of competitor brands with at least one store reaching it. It is fixed in a scenario,
and the operator's share of a hex is `demand ÷ (1 + k)` if it serves the hex.

In [ ]:
def place_network(C: Case, counts: dict[str, int] = BRAND_COUNTS, seed: int = SEED) -> dict[str, list[int]]:
    rng = np.random.default_rng(seed)
    reached = np.bincount(C.ps, weights=C.demand[C.ph], minlength=len(C.sites))
    chosen = rng.choice(len(C.sites), size=sum(counts.values()), replace=False, p=reached / reached.sum())
    labels = rng.permutation(np.repeat(list(counts), list(counts.values())))
    return {b: sorted(chosen[labels == b].tolist()) for b in counts}

def with_network(C: Case, stores: dict[str, list[int]], cap_int: int | None = None) -> Case:
    comp = [b for b in stores if b != "Operator"]
    hit = np.zeros((len(comp), len(C.hexes)), bool)
    for i, b in enumerate(comp):
        hit[i, C.ph[np.isin(C.ps, stores[b])]] = True
    k = hit.sum(axis=0)
    occupied = np.array(sorted(s for ss in stores.values() for s in ss))
    assert len(occupied) == len(set(occupied)), "two stores on one site"
    return replace(C, stores=stores, k=k, share_int=np.round(C.demand / (1 + k) * SCALE).astype(np.int64),
                   new_sites=np.setdiff1d(np.arange(len(C.sites)), occupied), cap_int=cap_int or C.cap_int)

cases = {res: with_network(C, place_network(C)) for res, C in bare.items()}
for res, C in cases.items():
    rows = []
    for b, ss in C.stores.items():
        hexes_b = np.unique(C.ph[np.isin(C.ps, ss)])
        rows.append({"brand": b, "stores": len(ss), "ranks of its sites": ", ".join(str(int(C.rank[s])) for s in ss),
                     "hexes reached": len(hexes_b), "demand reached (orders/day)": round(C.demand[hexes_b].sum())})
    print(f"res {res}: {len(C.hexes)} hexes; hexes by k (competitor brands reaching): "
          + ", ".join(f"k={i}: {int((C.k == i).sum())}" for i in range(3)) + f"; sites open for a new store: {len(C.new_sites)}")
    display(pd.DataFrame(rows).style.hide(axis="index"))

## The model

For a scenario, let `share_h = demand_h ÷ (1 + k_h)` (hundredths of an order, integers). Decision variables:

- `y_s ∈ {0,1}` for each candidate site without a store: open a new store there; `Σ y_s = N`. The operator's existing
  stores are open and fixed.
- `x_{s,h} ∈ {0,1}` for each reachable pair of an openable site and a hex: the operator serves hex *h* from store *s*.

Constraints: `Σ_s x_{s,h} ≤ 1` for each hex; `x_{s,h} ≤ y_s` for a new site; `Σ_h share_h · x_{s,h} ≤ capacity` for each
store. Objective: maximise `Σ share_h · x_{s,h}`, the operator's orders a day. Hexes a competitor reaches but the
operator does not are lost to competitors.

**Written exactly like that (the plain model), CP-SAT cannot solve it in a useful time** (measured below). The linear
relaxation is weak, so the bound barely moves from "serve every hex". The model used here is the same problem with fewer
variables. If everything a site could ever reach, `W_s = Σ_h share_h` over its catchment, is within capacity, then the
capacity constraint of that site can never bind: any hex it reaches can be assigned to it, whatever else is assigned. For
such a **free** site, the model needs no `x`, only whether each hex is covered: `c_h ≤ Σ y_s` over the free new sites
reaching *h*, and `c_h = 1` if a free existing store reaches it. The `x_{s,h}` and capacity constraints are kept for the
sites with `W_s > capacity`, which are the only ones where capacity can bind, and `c_h + Σ_s x_{s,h} ≤ 1` per hex. At
the default capacity no site in the slice is constrained, so the model is maximum coverage; with a lower capacity
lever the constrained sites appear (tested below). This is exact, not a heuristic: it has the same optimum as the plain
model. Two checks below: every plan's objective must equal what the evaluator, which solves the assignment from
scratch for the chosen stores, serves; and, at a capacity that binds, the two models must reach the same optimum.

In [ ]:
def solve_plan(C: Case, N: int, new_sites: np.ndarray | None = None, pair_model: bool = False,
               time_limit: float = TIME_LIMIT) -> dict[str, str | float | list[int]]:
    """Choose N new sites from new_sites (default: all open ones) to maximise the operator's orders.
    pair_model=True builds the plain model (x for every pair); otherwise only sites with W_s > capacity get x."""
    t0 = time.perf_counter()
    new = C.new_sites if new_sites is None else new_sites
    op = np.array(C.stores["Operator"])
    is_new, is_op = np.zeros(len(C.sites), bool), np.zeros(len(C.sites), bool)
    is_new[new], is_op[op] = True, True
    keep = (is_new | is_op)[C.ps]
    ps, ph = C.ps[keep], C.ph[keep]
    reach_w = np.bincount(ps, weights=C.share_int[ph], minlength=len(C.sites))      # W_s
    constrained = np.ones(len(C.sites), bool) if pair_model else reach_w > C.cap_int
    model = cp_model.CpModel()
    y = {int(s): model.NewBoolVar(f"y{s}") for s in new}
    model.Add(sum(y.values()) == N)
    terms, coeffs, const = [], [], 0
    x_at = defaultdict(list)
    for h, sites_h in pd.Series(ps).groupby(ph):
        sites_h = sites_h.to_numpy()
        w = int(C.share_int[h])
        free = sites_h[~constrained[sites_h]]
        if is_op[free].any():               # an existing store that cannot hit capacity covers this hex
            const += w
            continue
        row = []
        if len(free):
            covered = model.NewBoolVar("")
            model.AddBoolOr([y[int(s)] for s in free]).OnlyEnforceIf(covered)
            row.append(covered)
        for s in sites_h[constrained[sites_h]]:
            xv = model.NewBoolVar("")
            if is_new[s]:
                model.AddImplication(xv, y[int(s)])
            x_at[int(s)].append((xv, w))
            row.append(xv)
        if len(row) > 1:
            model.AddAtMostOne(row)
        terms += row
        coeffs += [w] * len(row)
    for lst in x_at.values():
        model.Add(cp_model.LinearExpr.WeightedSum([v for v, _ in lst], [w for _, w in lst]) <= C.cap_int)
    model.Maximize(cp_model.LinearExpr.WeightedSum(terms, coeffs) + const)
    t1 = time.perf_counter()
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_workers = WORKERS
    solver.parameters.random_seed = SOLVER_SEED
    status = solver.Solve(model)
    t2 = time.perf_counter()
    ok = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    return {"status": solver.StatusName(status), "chosen": sorted(s for s in y if solver.Value(y[s])) if ok else [],
            "obj": solver.ObjectiveValue() / SCALE, "bound": solver.BestObjectiveBound() / SCALE,
            "pairs": len(ps), "new_sites": len(new), "hexes": int(len(np.unique(ph))),
            "constrained sites": int(constrained[np.unique(ps)].sum()),
            "build_s": t1 - t0, "solve_s": t2 - t1, "total_s": t2 - t0}

## One evaluator for every plan

Every plan is scored by the same function, so the comparison does not depend on how a plan was found. Given the
operator's open stores, `evaluate` solves the assignment of hexes to stores from scratch: maximise the orders served
within capacity, one store per hex, solved to optimality. Then, among the assignments that serve those orders, the one
with the smaller total travel time (the tie-break in the assumptions), in a second solve with a 5 s limit, so the
tie-break is best effort and the orders served are exact. The tie-break matters
only for the split of a plan's orders into new and cannibalised demand; with ample capacity many assignments serve the
same orders, and "nearest store" is the natural way to choose between them. It returns the owner of each hex (a site
position, or -1) and the orders served.

## The greedy baseline

Add the candidate site with the largest marginal demand, one at a time, until N. The marginal of site *s* is
`min(capacity, Σ share_h` over hexes *s* reaches that the current assignment leaves unserved)`: hexes already served by an
open store do not count, and a site cannot gain more than a store can serve. "Unserved" comes from the evaluator run on
the stores opened so far, so a hex an open store reaches but cannot serve for want of capacity still counts. This is the
approximate marginal, not the exact change in orders (which would also count hexes a new store takes over from an
existing one and gives back capacity to others), and it is cheap: one evaluation per step. Ties go to the lower rank, the
busier site. The final score of the greedy plan comes from the shared evaluator, not from the greedy's own sums.

In [ ]:
def evaluate(C: Case, open_sites: list[int]) -> tuple[np.ndarray, int]:
    """Best assignment of hexes to the operator's open stores within capacity. Returns (owner per hex, orders x 100)."""
    keep = np.isin(C.ps, list(open_sites))
    ps, ph, pm = C.ps[keep], C.ph[keep], C.pm[keep]
    w = C.share_int[ph]
    model = cp_model.CpModel()
    x = [model.NewBoolVar("") for _ in range(len(ps))]
    for _, js in pd.Series(np.arange(len(ps))).groupby(ph):
        if len(js) > 1:
            model.AddAtMostOne([x[j] for j in js])
    for _, js in pd.Series(np.arange(len(ps))).groupby(ps):
        js = js.to_numpy()
        model.Add(cp_model.LinearExpr.WeightedSum([x[j] for j in js], [int(w[j]) for j in js]) <= C.cap_int)
    served = cp_model.LinearExpr.WeightedSum(x, [int(v) for v in w])
    model.Maximize(served)                      # stage 1: orders served; always solved to optimality
    solver = cp_model.CpSolver()
    solver.parameters.num_workers = 1
    solver.parameters.random_seed = SOLVER_SEED
    solver.parameters.max_time_in_seconds = 30
    status = solver.Solve(model)
    assert status == cp_model.OPTIMAL, solver.StatusName(status)
    best = round(solver.ObjectiveValue())
    for v in x:
        model.AddHint(v, solver.Value(v))
    model.Add(served == best)                   # stage 2: the same orders, the least total travel time (hundredths of a minute)
    model.Minimize(cp_model.LinearExpr.WeightedSum(x, [round(float(m) * 100) for m in pm]))
    solver.parameters.max_time_in_seconds = 5   # best effort: stage 1 fixes the orders, this only picks among ties
    status = solver.Solve(model)
    assert status in (cp_model.OPTIMAL, cp_model.FEASIBLE), solver.StatusName(status)
    owner = np.full(len(C.hexes), -1)
    for j in np.flatnonzero([solver.Value(v) for v in x]):
        owner[ph[j]] = ps[j]
    assert int(C.share_int[owner >= 0].sum()) == best
    return owner, best

def greedy_plan(C: Case, N: int) -> list[int]:
    opened, picked, cands = list(C.stores["Operator"]), [], C.new_sites.copy()
    for _ in range(N):
        owner, _ = evaluate(C, opened)
        unserved = np.where(owner < 0, C.share_int, 0)
        value = np.minimum(np.bincount(C.ps, weights=unserved[C.ph], minlength=len(C.sites))[cands], C.cap_int)
        s = int(cands[np.argmax(value)])      # first maximum: the lowest rank
        picked.append(s)
        opened.append(s)
        cands = cands[cands != s]
    return picked

## Does the model solve? The plain model against the one used

Res 9, N = 3, the default scenario, on the placeholder network. Both models are the same problem; the plain one has an
`x` for each of the slice's pairs. The plain model gets 20 s here, not the 60 s limit, to keep the notebook's run time down.
The bound is the solver's best proven upper bound on the operator's orders a day, counting the existing stores.

In [ ]:
rows = []
for label, kw in [("plain (x for every pair)", dict(pair_model=True, time_limit=20)), ("used (x only where capacity can bind)", {})]:
    for res in (9, 8):
        r = solve_plan(cases[res], N_SLICE, **kw)
        rows.append({"model": label, "res": res, "pairs": r["pairs"], "sites with x": r["constrained sites"], "status": r["status"],
                     "orders/day found": round(r["obj"], 1), "proven bound": round(r["bound"], 1),
                     "gap": f"{(r['bound'] - r['obj']) / r['obj']:.1%}", "build s": round(r["build_s"], 2), "solve s": round(r["solve_s"], 2)})
display(pd.DataFrame(rows).style.hide(axis="index"))
print(f"load average at this point: {os.getloadavg()[0]:.1f} on {os.cpu_count()} cores")

## The plans

Existing-only (the operator's own stores, no new ones), greedy and CP-SAT, at res 9, N = 3, scored by the shared
evaluator. The CP-SAT objective is checked against the evaluator: for the stores CP-SAT chose, the evaluator must
serve the same orders. **Metrics**, in orders a day, with the operator's potential on a hex defined as its share,
`demand ÷ (1 + k)`:

- **served**: operator orders the assignment serves; split into existing stores and new stores.
- **lost, over capacity (a)**: potential on hexes some open operator store reaches but the assignment does not serve.
- **lost, unreached (b)**: potential on hexes no open operator store reaches.
- Served + (a) + (b) = the operator's potential on the whole slice, the same for every plan.
- **to competitors**: on hexes an operator store reaches, the competitors' part (demand − share); on hexes it does
  not reach, all the demand when some competitor reaches the hex. **To no brand**: demand on hexes nobody reaches.
  Served + (a) + competitors + no brand = the slice's demand.
- For each new store: **new** demand is orders on hexes it serves that no operator store served in the existing-only plan;
  **cannibalised** is orders on hexes it serves that an existing operator store served in the existing-only plan.

In [ ]:
def plan_metrics(C: Case, picked: list[int], base_owner: np.ndarray) -> tuple[dict[str, float], pd.DataFrame, np.ndarray]:
    open_ = list(C.stores["Operator"]) + list(picked)
    owner, served_int = evaluate(C, open_)
    share = C.share_int / SCALE
    got = owner >= 0
    hit = np.zeros(len(C.hexes), bool)
    hit[C.ph[np.isin(C.ps, open_)]] = True
    from_old = share[got & np.isin(owner, C.stores["Operator"])].sum()
    m = {
        "operator stores": len(open_),
        "orders served": share[got].sum(),
        "  from existing stores": from_old,
        "  from new stores": share[got].sum() - from_old,
        "lost, over capacity (a)": share[hit & ~got].sum(),
        "lost, unreached (b)": share[~hit].sum(),
        "operator potential (served + a + b)": share.sum(),
        "to competitors": (C.demand - share)[hit].sum() + C.demand[~hit & (C.k > 0)].sum(),
        "to no brand": C.demand[~hit & (C.k == 0)].sum(),
    }
    assert abs(m["orders served"] + m["lost, over capacity (a)"] + m["to competitors"] + m["to no brand"] - C.demand.sum()) < 1e-6
    new_rows = []
    for s in picked:
        mine = owner == s
        new_rows.append({"site rank": int(C.rank[s]), "hexes served": int(mine.sum()), "orders served": share[mine].sum(),
                         "new": share[mine & (base_owner < 0)].sum(), "cannibalised": share[mine & (base_owner >= 0)].sum()})
    return m, pd.DataFrame(new_rows), owner

def run_plans(C: Case, N: int) -> tuple[dict[str, tuple[dict[str, float], pd.DataFrame, np.ndarray]], dict[str, str | float | list[int]], list[int]]:
    base_owner, _ = evaluate(C, C.stores["Operator"])
    r = solve_plan(C, N)
    g = greedy_plan(C, N)
    out = {}
    for name, picked in [("existing only", []), ("greedy", g), ("CP-SAT", r["chosen"])]:
        out[name] = plan_metrics(C, picked, base_owner)
    # the CP-SAT objective must equal what the evaluator serves for the chosen stores
    assert r["status"] == "OPTIMAL" and abs(r["obj"] - out["CP-SAT"][0]["orders served"]) < 0.005, (r["status"], r["obj"], out["CP-SAT"][0]["orders served"])
    return out, r, g

plans = {}
for res in (9, 8):
    plans[res] = run_plans(cases[res], N_SLICE)
    out, r, g = plans[res]
    print(f"res {res}: CP-SAT {r['status']}, bound {r['bound']:.2f}, objective {r['obj']:.2f}; chosen site ranks "
          f"{[int(cases[res].rank[s]) for s in r['chosen']]}; greedy site ranks {[int(cases[res].rank[s]) for s in g]}")
    display(pd.DataFrame({name: m for name, (m, _, _) in out.items()}).round(1))
    for name in ("greedy", "CP-SAT"):
        print(f"res {res}, {name}, per new store:")
        display(out[name][1].round(1).style.hide(axis="index"))
    gs, cs = out["greedy"][0]["orders served"], out["CP-SAT"][0]["orders served"]
    print(f"res {res}: CP-SAT serves {cs - gs:+.1f} orders/day against greedy ({(cs - gs) / gs:+.2%}); "
          f"existing-only {out['existing only'][0]['orders served']:.1f}")

How much the CP-SAT result depends on this one network: the same comparison, res 9, N = 3, for ten placements of the
placeholder network (seeds `SEED` to `SEED + 9`; the first is the network above).

In [ ]:
rows = []
for seed in range(SEED, SEED + 10):
    C = with_network(bare[9], place_network(bare[9], seed=seed))
    r = solve_plan(C, N_SLICE)
    g = greedy_plan(C, N_SLICE)
    op = C.stores["Operator"]
    gs, cs, e0 = (evaluate(C, op + p)[1] / SCALE for p in (g, r["chosen"], []))
    rows.append({"seed": seed, "existing only": e0, "greedy": gs, "CP-SAT": cs, "status": r["status"],
                 "CP-SAT vs greedy": cs - gs, "%": (cs - gs) / gs * 100, "same sites": set(g) == set(r["chosen"])})
seeds = pd.DataFrame(rows)
display(seeds.round(1).style.hide(axis="index"))
alt_seed = int(seeds.loc[seeds["CP-SAT vs greedy"].idxmax(), "seed"])     # the network where CP-SAT gains most, drawn below
print(f"CP-SAT better in {int((seeds['CP-SAT vs greedy'] > 0.005).sum())} of {len(seeds)}, tied in {int((seeds['CP-SAT vs greedy'].abs() <= 0.005).sum())}, "
      f"worse in {int((seeds['CP-SAT vs greedy'] < -0.005).sum())}; mean {seeds['%'].mean():+.2f}%, largest {seeds['%'].max():+.2f}%")

## Figures

The two plans on the slice at res 9, for the network above (top row) and for the one of the ten seeds where CP-SAT gains
most over greedy (bottom row). Hex shading is demand (orders a day per hex, before the brand split). Stores are
marked by brand; the new stores are stars. Outlines show where the operator's orders come from: blue, hexes served by an
existing operator store; vermillion, hexes a new store wins fresh; purple, hexes a new store takes from an existing store
(cannibalised). **The network is a placeholder**: counts and positions are not real stores.

In [ ]:
def hex_poly(c: str) -> Polygon:
    return Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)])

C9 = cases[9]
gdf = gpd.GeoDataFrame({"demand": C9.demand}, geometry=[hex_poly(c) for c in C9.hexes], crs=4326)
site_xy = pd.read_parquet(EXPLORE / "candidates-r9.parquet").loc[C9.sites, ["lon", "lat"]].to_numpy()
BRAND_STYLE = {"Operator": ("#0072B2", "s"), "Competitor A": ("#E69F00", "o"), "Competitor B": ("#009E73", "D")}
NEW_C, CANN_C, OLD_C = "#D55E00", "#7B3F9E", "#0072B2"
VMAX = np.percentile(C9.demand, 99)

def draw_plan(ax: Axes, C: Case, plan_set: tuple[dict[str, tuple[dict[str, float], pd.DataFrame, np.ndarray]], dict[str, str | float | list[int]], list[int]], name: str, tag: str) -> None:
    out, r, g = plan_set
    m, _, owner = out[name]
    base_owner = out["existing only"][2]
    picked = {"greedy": g, "CP-SAT": r["chosen"]}[name]
    gdf.plot(column="demand", ax=ax, cmap="Greys", linewidth=0.1, edgecolor="white", vmin=0, vmax=VMAX)
    got = owner >= 0
    old = got & np.isin(owner, C.stores["Operator"])
    fresh = got & np.isin(owner, picked) & (base_owner < 0)
    cann = got & np.isin(owner, picked) & (base_owner >= 0)
    for mask, colour, lw in [(old, OLD_C, 0.8), (fresh, NEW_C, 1.6), (cann, CANN_C, 1.6)]:
        if mask.any():
            gdf[mask].boundary.plot(ax=ax, color=colour, linewidth=lw)
    for b, (colour, marker) in BRAND_STYLE.items():
        xy = site_xy[C.stores[b]]
        ax.scatter(xy[:, 0], xy[:, 1], s=90, c=colour, marker=marker, edgecolor="black", linewidth=0.8, zorder=5)
    xy = site_xy[picked]
    ax.scatter(xy[:, 0], xy[:, 1], s=420, c=NEW_C, marker="*", edgecolor="black", linewidth=1.0, zorder=6)
    ax.set_aspect(1 / np.cos(np.radians(28.44)))
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    ax.set_title(f"{tag}, {name}: {m['orders served']:,.0f} orders/day", fontsize=11)

C_alt = with_network(bare[9], place_network(bare[9], seed=alt_seed))
plans_alt = run_plans(C_alt, N_SLICE)
fig, axes = plt.subplots(2, 2, figsize=(13, 13), constrained_layout=True)
for j, name in enumerate(("greedy", "CP-SAT")):
    draw_plan(axes[0, j], C9, plans[9], name, f"network seed {SEED}")
    draw_plan(axes[1, j], C_alt, plans_alt, name, f"network seed {alt_seed}")
handles = [Line2D([], [], marker=mk, ls="", mfc=c, mec="black", ms=9, label=f"{b} (existing, placeholder)") for b, (c, mk) in BRAND_STYLE.items()]
handles += [Line2D([], [], marker="*", ls="", mfc=NEW_C, mec="black", ms=16, label="new store")]
handles += [Line2D([], [], color=c, lw=2, label=l) for c, l in [(OLD_C, "hexes of an existing operator store"), (NEW_C, "hexes a new store wins fresh"), (CANN_C, "hexes a new store takes (cannibalised)")]]
fig.legend(handles=handles, loc="outside lower center", ncol=4, fontsize=9, frameon=False)
sm = plt.cm.ScalarMappable(cmap="Greys", norm=plt.Normalize(vmin=0, vmax=VMAX))
fig.colorbar(sm, ax=axes, shrink=0.5, label="demand, orders per day per hex (before the brand split)")
fig.suptitle(f"Plans on the slice, res 9, N = {N_SLICE}, default scenario. Top: the main network; bottom: the network where CP-SAT gains most")
fig.savefig(FIGS / "02-plans.png", dpi=120, bbox_inches="tight")
plt.show()

![Greedy and CP-SAT plans on the slice](figures/02-plans.png)

In [ ]:
out9 = plans[9][0]
names = ["existing only", "greedy", "CP-SAT"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True, gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
parts = [("from existing stores", "  from existing stores", "#0072B2"), ("from new stores", "  from new stores", "#D55E00"),
         ("lost, over capacity (a)", "lost, over capacity (a)", "#999999"), ("lost, unreached (b)", "lost, unreached (b)", "#CCCCCC")]
bottom = np.zeros(len(names))
for label, key, colour in parts:
    vals = np.array([out9[n][0][key] for n in names])
    ax.bar(names, vals, bottom=bottom, color=colour, label=label, edgecolor="white")
    for i, v in enumerate(vals):
        if v > 150:
            ax.text(i, bottom[i] + v / 2, f"{v:,.0f}", ha="center", va="center", fontsize=9, color="white" if colour in ("#0072B2", "#D55E00") else "black")
    bottom += vals
ax.set_ylabel("operator orders per day (potential = demand ÷ (1 + k))")
ax.set_title(f"The operator's potential on the slice: {out9['CP-SAT'][0]['operator potential (served + a + b)']:,.0f} orders/day")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, fontsize=9, frameon=False)
ax = axes[1]
labels, new, cann = [], [], []
for n in ("greedy", "CP-SAT"):
    df = out9[n][1]
    for i, row in df.iterrows():
        labels.append(f"{n}\nsite rank {int(row['site rank'])}"); new.append(row["new"]); cann.append(row["cannibalised"])
pos = np.arange(len(labels))
ax.bar(pos, new, color=NEW_C, label="new demand")
ax.bar(pos, cann, bottom=new, color=CANN_C, label="cannibalised")
ax.set_xticks(pos, labels, fontsize=8)
ax.set_ylabel("orders per day")
ax.set_ylim(0, 2100)
ax.set_title("Each new store: new against cannibalised orders")
ax.legend(fontsize=9)
fig.suptitle("Plan metrics, slice, res 9, N = 3 (placeholder network)")
fig.savefig(FIGS / "02-metrics.png", dpi=120, bbox_inches="tight")
plt.show()

![Plan metrics: served and lost orders, and new against cannibalised demand per new store](figures/02-metrics.png)

Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; re-running updates the printed
tables, not these):

**The plain model does not solve; the one used does.** Res 9, N = 3, default scenario, same network:

| model | res | status | orders/day found | proven bound | gap | build + solve |
|---|---:|---|---:|---:|---:|---:|
| plain (`x` for every pair, 547 sites with `x`) | 9 | FEASIBLE after 20 s | 5,085.5 | 13,914.3 | 174% | 0.54 + 20.25 s |
| used (`x` only where capacity can bind, 0 sites) | 9 | OPTIMAL | 5,599.4 | 5,599.4 | 0% | 0.04 + 0.17 s |
| plain | 8 | OPTIMAL | 4,944.2 | 4,944.2 | 0% | 0.10 + 0.05 s |
| used | 8 | OPTIMAL | 4,944.2 | 4,944.2 | 0% | 0.00 + 0.01 s |

The plain model's answer after 20 s is 9% below the optimum in this run (4,394 to 5,086 over three runs of the notebook, 22% to 9% below) and its bound is the trivial one (every hex served). At
the default capacity the largest catchment holds 1,981 orders a day of demand (res 9; 1,714 at res 8; the operator's share is
at most that) against a capacity of 2,364, so no site is capacity-constrained and the used model is maximum coverage.

**Network (placeholder, seed 2026), res 9:** operator 3 stores, Competitor A 4, Competitor B 3. Hexes by number of competitor
brands reaching them: k = 0: 517, k = 1: 182, k = 2: 14. The operator's three stores reach 65 hexes (1,541 orders a day of
demand), A's four reach 119 (2,791), B's three reach 91 (2,355). 544 of the 554 candidate sites are open for a new store.
Res 8: 3, 4, 3 stores; k = 0: 69, k = 1: 30, k = 2: 3; 74 of 84 open.

**Plans, res 9, N = 3, scored by the shared evaluator (orders a day):**

| | existing only | greedy | CP-SAT |
|---|---:|---:|---:|
| operator stores | 3 | 6 | 6 |
| **orders served** | **1,089.7** | **5,599.4** | **5,599.4** |
| from existing stores | 1,089.7 | 731.0 | 731.0 |
| from new stores | 0 | 4,868.4 | 4,868.4 |
| lost, over capacity (a) | 0 | 0 | 0 |
| lost, unreached (b) | 12,986.9 | 8,477.2 | 8,477.2 |
| operator potential (served + a + b) | 14,076.6 | 14,076.6 | 14,076.6 |
| to competitors | 4,328.2 | 4,177.0 | 4,177.0 |
| to no brand | 11,104.9 | 6,746.5 | 6,746.5 |

(The slice's demand is 16,522.8: served + (a) + competitors + no brand adds up to it in each column.) **On this network
CP-SAT and greedy pick the same three sites and tie.** CP-SAT's bound equals its objective (OPTIMAL), so 5,599.4 is the
best three-store plan on this network, and greedy found it. Per new store (the same for both plans), site rank in the area's places
count:

| site rank | hexes served | orders served | new | cannibalised |
|---:|---:|---:|---:|---:|
| 314 | 64 | 1,722.3 | 1,722.3 | 0.0 |
| 56 | 74 | 1,757.4 | 1,720.2 | 37.1 |
| 86 | 54 | 1,388.8 | 1,067.1 | 321.7 |

Net gain over the existing-only plan is 4,509.7 orders a day: the new stores serve 4,868.4, and the existing stores lose
358.7 of their 1,089.7 to them (the cannibalised 358.8, up to rounding). Each new store serves 1,390 to 1,760 orders a day,
close to `ORDERS_PER_STORE_DAY` (1,576), and none is near capacity (2,364). Res 8: both plans again tie, at 4,944.2 against 1,792.7
existing-only (sites of ranks 7, 47, 83; no cannibalisation; each new store serves 5 to 7 hexes).

**Across ten network placements** (seeds 2026 to 2035, res 9, N = 3; all CP-SAT runs OPTIMAL): CP-SAT is better in 2,
tied in 8 and never worse. It serves 4.6% more than greedy on seed 2027 (5,691.9 against 5,440.1) and 2.5% more on 2031
(5,309.9 against 5,182.4); the mean advantage is 0.71%.

## Solve time

What the 10 s bar needs: model build plus solve, for the default scenario on the full area (N = 10). Only the slice has a
travel-time table, so the full-area time has to be extrapolated from slice runs. The method, with its limits:

1. **Slice, N = 3, res 9 and res 8** (the plan above): wall time of `solve_plan` (build the model in Python, then
   solve), median of three runs.
2. **A grid of sizes.** Candidate sites for new stores are the slice's candidates in the top X% of the area's hexes by places
   count, for each X in the grid (as in `01-reach`), and N in {3, 5, 10}. The operator's existing stores always stay.
   The size measure is the number of (site, hex) pairs in the model. The median of three runs per cell (the minimum and maximum are shown), time limit 60 s, status recorded.
3. **A log-log fit** of total time on pairs, at N = 10, extrapolated to the full area's default-scenario pair count
   from `01-reach` (54,246 at res 9, 1,105 at res 8), and to X = 10% at res 9 (54,246 × 486 / 1,458 pairs).
4. **A direct check at full-area pair count**: the slice instance copied three times side by side (independent copies
   with their own stores), which has about the full area's pairs, with N = 10 (median of three) and N = 30 (10 per copy, one run).

The machine is shared with other jobs, so times move from run to run; the load average is printed.

In [ ]:
def time_case(C: Case, N: int, new: np.ndarray | None = None, reps: int = 1) -> dict[str, str | float | list[int]]:
    runs = [solve_plan(C, N, new) for _ in range(reps)]
    r = dict(sorted(runs, key=lambda r: r["total_s"])[len(runs) // 2])
    r["min_s"], r["max_s"] = min(x["total_s"] for x in runs), max(x["total_s"] for x in runs)
    return r

print(f"load average at the start: {os.getloadavg()[0]:.1f} on {os.cpu_count()} cores")
rows = []
for res in (9, 8):
    C = cases[res]
    for N in (N_SLICE, N_AREA):
        r = time_case(C, N, reps=3)
        rows.append({"res": res, "N": N, "new sites": r["new_sites"], "pairs": r["pairs"], "hexes": r["hexes"], "status": r["status"],
                     "build s": r["build_s"], "solve s": r["solve_s"], "total s": r["total_s"], "min s": r["min_s"], "max s": r["max_s"]})
headline = pd.DataFrame(rows)
display(headline.round(2).style.hide(axis="index"))

In [ ]:
grid_rows = []
for res in (9, 8):
    C = cases[res]
    for X in X_GRID:
        new = C.new_sites[C.rank[C.new_sites] < math.ceil(X / 100 * C.n_area_hexes)]
        for N in N_GRID:
            if len(new) < N:
                continue
            r = time_case(C, N, new, reps=3)
            op = C.stores["Operator"]
            grid_rows.append({"res": res, "X %": X, "new sites": len(new), "pairs": r["pairs"], "N": N, "status": r["status"],
                              "build s": r["build_s"], "solve s": r["solve_s"], "total s": r["total_s"], "min s": r["min_s"], "max s": r["max_s"], "orders": r["obj"],
                              "greedy orders": evaluate(C, op + greedy_plan(replace(C, new_sites=new), N))[1] / SCALE if X == 30 else np.nan})
grid = pd.DataFrame(grid_rows)
display(grid.round(2).style.hide(axis="index"))

Fit and extrapolation. `log10(total s) = a + b · log10(pairs)`, fitted on the N = 10 rows of each resolution, and a
second fit over all N with `+ c · log10(N)`. The prediction is at the full area's pair count. A fit through a handful of
points that span less than a decade is a rough guide only; the caveats follow the results.

In [ ]:
fit_rows, fits = [], {}
for res in (9, 8):
    g = grid[grid.res == res]
    g10 = g[g.N == N_AREA]
    b, a = np.polyfit(np.log10(g10.pairs), np.log10(g10["total s"]), 1)
    A = np.column_stack([np.ones(len(g)), np.log10(g.pairs), np.log10(g.N)])
    coef, *_ = np.linalg.lstsq(A, np.log10(g["total s"]), rcond=None)
    targets = {"full area": FULL_AREA[res]["pairs"]}
    if res == 9:
        targets["X = 10%"] = round(FULL_AREA[9]["pairs"] * 486 / 1458)
    for name, pairs in targets.items():
        fit_rows.append({"res": res, "target": name, "pairs": pairs, "N": N_AREA,
                         "N = 10 fit: slope": b, "predicted s (N = 10 fit)": 10 ** (a + b * np.log10(pairs)),
                         "all-N fit: pairs, N exponents": f"{coef[1]:.2f}, {coef[2]:.2f}",
                         "predicted s (all-N fit)": 10 ** (coef[0] + coef[1] * np.log10(pairs) + coef[2] * np.log10(N_AREA))})
    fits[res] = (a, b)
display(pd.DataFrame(fit_rows).round(2).style.hide(axis="index"))
print("slowest grid run:", grid.loc[grid["total s"].idxmax(), ["res", "X %", "N", "status", "total s"]].to_dict())
print("statuses:", grid.status.value_counts().to_dict())

In [ ]:
def tile(C: Case, copies: int) -> Case:
    """copies of the slice instance side by side, independent of each other (a stress test of size, not a real area)."""
    nS, nH = len(C.sites), len(C.hexes)
    off = np.arange(copies)
    stores = {b: [s + nS * i for i in range(copies) for s in v] for b, v in C.stores.items()}
    occupied = np.array(sorted(s for v in stores.values() for s in v))
    return replace(C, hexes=pd.RangeIndex(copies * nH), demand=np.tile(C.demand, copies), sites=pd.RangeIndex(copies * nS),
                   rank=np.tile(C.rank, copies), ps=(C.ps[None, :] + nS * off[:, None]).ravel(),
                   ph=(C.ph[None, :] + nH * off[:, None]).ravel(), pm=np.tile(C.pm, copies), stores=stores,
                   k=np.tile(C.k, copies), share_int=np.tile(C.share_int, copies),
                   new_sites=np.setdiff1d(np.arange(copies * nS), occupied))

rows = []
for res, copies in [(9, 3), (8, 3)]:
    T = tile(cases[res], copies)
    for N in (N_AREA, N_AREA * copies):
        r = time_case(T, N, reps=3 if N == N_AREA else 1)
        rows.append({"res": res, "copies": copies, "hexes": len(T.hexes), "new sites": r["new_sites"], "pairs": r["pairs"], "N": N,
                     "status": r["status"], "build s": r["build_s"], "solve s": r["solve_s"], "total s": r["total_s"], "min s": r["min_s"], "max s": r["max_s"]})
tiled = pd.DataFrame(rows)
display(tiled.round(2).style.hide(axis="index"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, res in zip(axes, (9, 8)):
    g = grid[grid.res == res]
    for N, marker in zip(N_GRID, "osD"):
        gn = g[g.N == N]
        ax.loglog(gn.pairs, gn["total s"], marker + "-", label=f"N = {N}")
    a, b = fits[res]
    xs = np.logspace(np.log10(g.pairs.min()), np.log10(FULL_AREA[res]["pairs"]), 50)
    ax.loglog(xs, 10 ** (a + b * np.log10(xs)), "k--", lw=1, label="fit at N = 10, extrapolated")
    ax.axvline(FULL_AREA[res]["pairs"], color="grey", lw=0.8, ls=":")
    ax.text(FULL_AREA[res]["pairs"], ax.get_ylim()[0], " full area", rotation=90, va="bottom", ha="right", fontsize=9, color="grey")
    ax.axhline(10, color="#D55E00", lw=1)
    ax.text(ax.get_xlim()[0], 10, " 10 s bar", va="bottom", color="#D55E00", fontsize=9)
    ax.set_xlabel("pairs in the model (site, hex), default scenario")
    ax.set_ylabel("build + solve, seconds")
    ax.set_title(f"Res {res}: slice runs, X from 2.5% to 30%")
    tl = tiled[(tiled.res == res) & (tiled.N == N_AREA)]
    ax.loglog(tl.pairs, tl["total s"], "kx", ms=10, mew=2, label="3 copies of the slice, N = 10")
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.legend(fontsize=9)
fig.savefig(FIGS / "02-solve-time.png", dpi=120, bbox_inches="tight")
plt.show()

![Solve time against model size, with the log-log fit and the full-area pair count](figures/02-solve-time.png)

CP-SAT against greedy at larger N on the slice (res 9 and res 8, all candidates, X = 30%), from the grid above and scored by the shared evaluator.

In [ ]:
cmp_rows = []
for res in (9, 8):
    C = cases[res]
    top = grid[(grid.res == res) & (grid["X %"] == 30)]
    for _, row in top.iterrows():
        cmp_rows.append({"res": res, "N": row.N, "greedy": row["greedy orders"], "CP-SAT": row.orders, "status": row.status,
                         "CP-SAT vs greedy": row.orders - row["greedy orders"], "%": (row.orders / row["greedy orders"] - 1) * 100})
display(pd.DataFrame(cmp_rows).round(2).style.hide(axis="index"))

Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; timings move with machine load,
and the machine was shared: load average 15 to 16 on 8 cores during these runs (23 to 28 in earlier runs of the same code), so they are probably pessimistic):

**Slice, `solve_plan` (build the model in Python, then solve), median of three, status OPTIMAL in every run:**

| res | N | pairs | build s | solve s | total s | (min, max) |
|---:|---:|---:|---:|---:|---:|---|
| 9 | 3 | 18,277 | 0.08 | 0.35 | 0.43 | 0.29, 0.69 |
| 9 | 10 | 18,277 | 0.06 | 1.20 | 1.26 | 1.14, 2.40 |
| 8 | 3 | 367 | 0.05 | 0.03 | 0.07 | 0.05, 0.08 |
| 8 | 10 | 367 | 0.01 | 0.10 | 0.11 | 0.06, 0.19 |

(Pairs are the default-scenario pairs of the sites the model may use: the slice's 544 open sites plus the operator's three
stores, 18,277 of the 18,504 pairs in the table.)

**Grid.** All 35 runs (res 9 and res 8, six values of X, N = 3, 5, 10; res 8's smallest X has too few sites for N = 10)
ended OPTIMAL. The slowest was 2.34 s (res 9, X = 20%, N = 10). Res 9 at N = 10, total seconds: 0.21 (2,147 pairs, X = 2.5%), 0.42 (4,106), 0.29 (7,765), 1.59 (10,953), 2.34 (14,236), 1.33 (18,277, X = 30%). Time is noisy and not monotone in pairs or in N. A cell's minimum and maximum over its three repeats differ by up to a factor of 2, and runs of the whole notebook differ more.

**Fit and extrapolation to the full area at N = 10** (`log10(total s) = a + b log10(pairs)`):

| | pairs | fit at N = 10 (slope) | all-N fit (exponents of pairs, N) |
|---|---:|---|---|
| res 9, full area | 54,246 | 5.6 s (1.05) | 3.1 s (0.69, 0.55) |
| res 9, X = 10% (54,246 × 486 / 1,458) | 18,082 | 1.8 s | 1.5 s |
| res 8, full area | 1,105 | 0.04 s (0.34) | 0.04 s (0.43, -0.24) |

**Direct check at the full area's pair count** (the slice instance copied three times, independent copies):

| res | hexes | pairs | N | total s | (min, max) | status |
|---:|---:|---:|---:|---:|---|---|
| 9 | 2,139 | 54,831 | 10 | 1.20 | 1.06, 1.43 | OPTIMAL |
| 9 | 2,139 | 54,831 | 30 | 18.8 (one run) | | OPTIMAL |
| 8 | 306 | 1,101 | 10 | 0.07 | 0.07, 0.11 | OPTIMAL |
| 8 | 306 | 1,101 | 30 | 0.11 (one run) | | OPTIMAL |

**CP-SAT against greedy at larger N** (slice, X = 30%, scored by the shared evaluator):

| res | N | greedy | CP-SAT | CP-SAT vs greedy |
|---:|---:|---:|---:|---|
| 9 | 3 | 5,599.4 | 5,599.4 | tie |
| 9 | 5 | 7,282.6 | 7,327.2 | +44.6 (+0.6%) |
| 9 | 10 | 10,324.5 | 10,650.7 | +326.2 (+3.2%) |
| 8 | 3 | 4,944.2 | 4,944.2 | tie |
| 8 | 5 | 6,652.0 | 6,703.0 | +51.0 (+0.8%) |
| 8 | 10 | 10,013.5 | 10,013.5 | tie |

**Is 10 s reachable?** At res 9, the evidence says yes, with a margin that the extrapolation cannot pin down: the slice at
N = 10 takes 1.3 s; the fit puts the full area (54,246 pairs) at 3.1 to 5.6 s; three copies of the slice with the full area's pair count take 1.2 s; X = 10% is about 1.5 to 1.8 s. Other runs of the same cells (one repeat per grid cell at first, then three) gave 2.7, 11.3, 7.1 and, at lower load, 1.9 s for the N = 10 fit, so the fit alone cannot rule out 10 s; the direct check argues it is not close. If res 9 did miss, cutting X before the resolution is the move the issue asks for: X = 10% cuts the pairs to a third. At res 8 it is not in question: well under a second. Res 8 does not reach the coverage target (`01-reach`: 96.0% of slice demand at
X = 30%), so the question is whether res 9 holds in stage 1, not whether res 8 would.

**Caveats of the extrapolation.** (1) The fit spans 2,147 to 18,277 pairs and is extrapolated a factor 3 beyond; the points are noisy (a factor 2 between repeats of a cell) and the fitted slope has been 0.9 to 1.4 across runs. (2) The slice is denser than the area: 26 candidate sites
reach each slice hex, against about 11 in the area (54,246 pairs over 4,859 hexes), which makes the slice's overlap, and
so its search, harder per pair; but the area has 7 times as many hexes, and neither the fit (which only sees pairs) nor
the copies (independent, identical, so very regular) capture how the budget N couples distant parts of the area. (3) N
matters: with N = 30 on three copies the time is 19 s (21 s in an earlier run), so if the service allows N well above 10 on the full area the bar
needs checking at that N. (4) Build time is Python and about linear in pairs (0.1 to 0.35 s here); the full-area model also needs its pairs in memory
(54,246 pairs, under a megabyte). (5) Machine load: see above; times on an idle machine would be lower. (6) CP-SAT's
parallel search is not deterministic across runs even with a seed, so a run's time is a sample.

## Capacity that binds

At the default capacity no site in the slice is constrained, so capacity never decides a plan. A one-off check that the
model handles the constrained case: capacity 1,000 orders a day (about 0.63 × the calibrated average, an arbitrary lower
value, not a scenario), res 9, N = 3, time limit 20 s, scored by the evaluator against greedy.

In [ ]:
C_low = with_network(bare[9], cases[9].stores, cap_int=100_000)
r = solve_plan(C_low, N_SLICE, time_limit=20)
g = greedy_plan(C_low, N_SLICE)
op = C_low.stores["Operator"]
print(f"capacity 1,000 orders/day: {r['constrained sites']} sites with W_s above capacity; CP-SAT {r['status']}, found {r['obj']:.1f}, bound {r['bound']:.1f}, "
      f"build {r['build_s']:.1f} s, solve {r['solve_s']:.1f} s")
print(f"scored by the evaluator: existing only {evaluate(C_low, op)[1] / SCALE:.1f}, greedy {evaluate(C_low, op + g)[1] / SCALE:.1f}, "
      f"CP-SAT {evaluate(C_low, op + r['chosen'])[1] / SCALE:.1f}")

Results from the run on 2026-10-29 (copied here as above): with capacity at 1,000 orders a day, 119 of the slice's sites have
`W_s` above capacity and need `x` variables. CP-SAT ended FEASIBLE after the 20 s limit, with 4,089.7 found and a proven
bound of 5,635.2 (a 38% gap). The evaluator scores existing-only 1,089.7, greedy 4,089.7, CP-SAT 4,089.7: no better than
greedy, and not proven optimal. So the formulation that makes the default scenario easy does not carry over to a capacity
that binds; the default capacity never binds in the slice (largest catchment's demand 1,981 against 2,364).

**Is the model used the same as the plain one where capacity binds?** It drops `x` only for sites that can never
reach capacity, so it should have the same optimum. A check where some sites do have `x`: res 8 at capacity 800 orders a
day (both models solve to optimality there, so their objectives must match), and res 9 at 1,700. There the plain model does
not finish in 20 s, so the check is weaker: the used model's objective must equal the evaluator's score for its own
plan, which catches a model that promises orders it cannot serve but not one that is too strict.

In [ ]:
rows = []
for res, cap in [(8, 800), (9, 1700)]:
    C = with_network(bare[res], cases[res].stores, cap_int=cap * SCALE)
    p, u = solve_plan(C, N_SLICE, pair_model=True, time_limit=20), solve_plan(C, N_SLICE, time_limit=20)
    ev = evaluate(C, C.stores["Operator"] + u["chosen"])[1] / SCALE
    assert u["constrained sites"] > 0, "no site has x: the check would be vacuous"
    assert u["status"] == "OPTIMAL" and abs(u["obj"] - ev) < 0.005, (u["status"], u["obj"], ev)
    if res == 8:   # both models solve to optimality here, so their optima must match
        assert p["status"] == "OPTIMAL" and abs(p["obj"] - u["obj"]) < 0.005, (p["status"], p["obj"], u["obj"])
    rows.append({"res": res, "capacity": cap, "sites with x": u["constrained sites"], "plain": f"{p['status']} {p['obj']:,.1f}",
                 "used": f"{u['status']} {u['obj']:,.1f}", "evaluator, used plan": round(ev, 1)})
display(pd.DataFrame(rows).style.hide(axis="index"))

Results from the run on 2026-10-29: res 8 at 800, 24 sites with `x`: both models OPTIMAL at 4,181.4, and the evaluator
agrees. Res 9 at 1,700, 4 sites with `x`: the used model is OPTIMAL at 5,556.2 (the evaluator agrees); the plain model
stops short after 20 s (5,062.1 and 4,804.0 in two runs; CP-SAT's parallel search varies). At lower capacities
the used model is not proven optimal in the time limit either (res 9 at 1,000, above).

## Results

All numbers from the run on 2026-10-29, in orders a day on the slice (demand 16,523 at res 9), on the placeholder network
(seed 2026), default scenario (promise 10, multiplier 1.4, capacity 2,364 per store):

| | res 9 | res 8 |
|---|---:|---:|
| operator existing-only | 1,089.7 | 1,792.7 |
| greedy, N = 3 | 5,599.4 | 4,944.2 |
| CP-SAT, N = 3 | 5,599.4 (OPTIMAL) | 4,944.2 (OPTIMAL) |
| CP-SAT against greedy, N = 3 | tie | tie |
| CP-SAT against greedy, N = 5 / N = 10 | +0.6% / +3.2% | +0.8% / tie |
| CP-SAT against greedy, 10 network seeds, N = 3 | better 2, tied 8, worse 0; mean +0.71%, best +4.6% | |
| slice solve time, N = 3 / N = 10 (build + solve) | 0.43 s / 1.26 s | 0.07 s / 0.11 s |
| full area, N = 10, extrapolated (fit) | 3.1 to 5.6 s | 0.04 s |
| full-area pair count, three slice copies, N = 10 | 1.2 s | 0.07 s |
| X = 10% (res 9), extrapolated | 1.5 to 1.8 s | |

- **CP-SAT beats greedy, but by a little at N = 3 and more as N grows.** On the main network they tie; across ten
  networks CP-SAT gains 0 to 4.6%; at N = 10 it gains 3.2%. Greedy is a good baseline for a handful of stores on this slice
  and falls behind as N rises. With ample capacity the problem is maximum coverage, where greedy is known to be good.
- **The cost of the lead is the formulation.** The model as first written (an `x` per pair) is not solvable in useful time, with a bound
  that does not move from the trivial one. The version used (no `x` for sites that cannot hit capacity) solves the
  default scenario to proven optimality in about a second on the slice. That version does not extend to a capacity
  that binds (tested at 1,000 orders a day: 38% gap after 20 s).
- **10 s looks reachable at res 9**, with an extrapolation of 3.1 to 5.6 s that is uncertain by a factor 2 to 3 (other runs gave 1.9 to 11.3 s), and a direct check at the full area's pair count of 1.2 s; X need not come down on this evidence, and res 8 is not needed for speed. It is not proven until the full-area table
  exists (stage 1).
- **Capacity never binds at the default** (the largest catchment holds 1,981 orders a day of demand, against a capacity of
  2,364), so over-capacity loss (a) is zero in every plan here. The capacity lever only matters below about 2,000.
- **Most of the slice is out of reach of everyone.** With the placeholder 10 stores, 11,105 of the slice's 16,523
  orders a day sit on hexes no brand reaches (6,747 after the operator adds three stores). The slice has about 10.5 stores'
  worth of orders, but a default-scenario catchment (4 to 6 km², about 1,000 to 2,000 orders) covers a small part of
  77.5 km², and the stores' catchments overlap. Sizing the network by orders and sizing it by reach disagree; stage 2 should
  decide which one the counts follow.

## Next

- **Stage 1** builds the full-area travel-time table, so the full-area solve time can be measured instead of
  extrapolated. Whether `X` has to come down depends on that number (decision log).
- **Stage 2** moves `solve_plan`, `evaluate` and `greedy_plan` into `src/catchment/` and replaces the placeholder network with the generator: counts sized from reported figures, source and date
  recorded for each. The comparison of CP-SAT and greedy and the new / cannibalised split depend on the network, so they
  should be re-read on it.
- **Snapping** (open in `01-reach`): about one short pair in seven is inflated by snapping onto the wrong carriageway.
  It changes which hexes a site reaches, and so the catchments the solver works with. Untried fixes are in #4.
- **Capacity that binds** is slow for the model used here (check above). The lever range for capacity, and a faster
  formulation or a warm start from the greedy plan for the constrained case, need a decision before the capacity
  lever goes in the service.
- **Inputs**: the four calibration inputs of `00-demand` were verified against primary sources on 2026-10-29, with no
  change in value; the capacity, the network counts and the capture share rest on them. Their remaining caveats
  (population projection, best-effort store count, Blinkit rate for every brand) are in `docs/decisions.md`.